# Data Modeling - Location & Feeder

### Location Table
The `location_silver` table is already complete in terms of transformations. The only remaining step is to add a primary key column.

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.location_gold (
  id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  barangay STRING,
  municipality STRING
);
INSERT INTO beneco_pipeline.gold.location_gold (barangay, municipality)
SELECT Barangay, Municipality FROM beneco_pipeline.silver.location_silver;

### Feeder Table
Same with the location table, `feeder_silver` table is already complete and only needs a primary id column.

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.feeder_gold (
  id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  name STRING
);
INSERT INTO beneco_pipeline.gold.feeder_gold (name)
SELECT Feeder FROM beneco_pipeline.silver.feeder_silver;

### Supply Junction Table
To represent the many-to-many relationship between the `feeder` and `location` tables, there needs to be a junction table that relates them. 

This junction table is created using the exploded feeder data from the loc_feeder_bronze table. Since each row in the exploded dataset represents a single feeder–location pair.

The table includes:
* `location_id` – a foreign key referencing the location table
* `feeder_id` – a foreign key referencing the feeder table

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.supply_gold (
  location_id BIGINT,
  feeder_id BIGINT,
  FOREIGN KEY (location_id) REFERENCES beneco_pipeline.gold.location_gold(id),
  FOREIGN KEY (feeder_id) REFERENCES beneco_pipeline.gold.feeder_gold(id)
);

INSERT INTO beneco_pipeline.gold.supply_gold (
SELECT 
  gl.id AS location_id,
  gf.id AS feeder_id
FROM (
  SELECT 
    Barangay,
    explode(split(Feeder, ', ')) AS Feeder
  FROM beneco_pipeline.bronze.loc_feeder_bronze
) AS lb
JOIN beneco_pipeline.gold.location_gold AS gl ON
  gl.Barangay = lb.Barangay
JOIN beneco_pipeline.gold.feeder_gold AS gf ON
  gf.name = lb.Feeder
ORDER BY location_id
);

In [0]:
%sql
SELECT * FROM beneco_pipeline.gold.supply_gold;